### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_5934/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory """
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

                all_documents.extend(documents)
                print (f"* Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 4 PDF files to process

Processing:langchain.pdf
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages
* Loaded 28 pages

Processing:masteringaiagents.pdf
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded 111 pages
* Loaded

In [3]:
all_pdf_documents

[Document(metadata={'producer': '3.0.36 (5.1.21)', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-21T18:15:49+00:00', 'title': 'LangChain Cheatsheet Carousel', 'moddate': '2026-06-21T20:20:17+02:00', 'source': '../data/pdf/langchain.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1', 'source_file': 'langchain.pdf', 'file_type': 'pdf'}, page_content='\uf5dc LANGCHAIN MASTERY\nThe Complete\nLangChain\nCheatsheet\nEvery component, every concept,\none real app built step-by-step.\nAI Coach John\n Follow \uf079 Repost'),
 Document(metadata={'producer': '3.0.36 (5.1.21)', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-21T18:15:49+00:00', 'title': 'LangChain Cheatsheet Carousel', 'moddate': '2026-06-21T20:20:17+02:00', 'source': '../data/pdf/langchain.pdf', 'total_pages': 28, 'page': 1

In [3]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunck
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")

    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)
chunks

Split 14613 documents into 25903 chunks

Example chunk:
Content: LANGCHAIN MASTERY
The Complete
LangChain
Cheatsheet
Every component, every concept,
one real app built step-by-step.
AI Coach John
 Follow  Repost...
Metadata:{'producer': '3.0.36 (5.1.21)', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-21T18:15:49+00:00', 'title': 'LangChain Cheatsheet Carousel', 'moddate': '2026-06-21T20:20:17+02:00', 'source': '../data/pdf/langchain.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1', 'source_file': 'langchain.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '3.0.36 (5.1.21)', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-21T18:15:49+00:00', 'title': 'LangChain Cheatsheet Carousel', 'moddate': '2026-06-21T20:20:17+02:00', 'source': '../data/pdf/langchain.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1', 'source_file': 'langchain.pdf', 'file_type': 'pdf'}, page_content='\uf5dc LANGCHAIN MASTERY\nThe Complete\nLangChain\nCheatsheet\nEvery component, every concept,\none real app built step-by-step.\nAI Coach John\n Follow \uf079 Repost'),
 Document(metadata={'producer': '3.0.36 (5.1.21)', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-21T18:15:49+00:00', 'title': 'LangChain Cheatsheet Carousel', 'moddate': '2026-06-21T20:20:17+02:00', 'source': '../data/pdf/langchain.pdf', 'total_pages': 28, 'page': 1

### Embedding and vectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/home/juan/PycharmProjects/agenticAICourse/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts:list[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts),embedding_dim) 
        """
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}") # shape = (text numbers, dimension)
        return embeddings

## initialice the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 320.52it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [2]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__ (self, collection_name: str = "pdf_documents", persist_directory: str= "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB Collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory= persist_directory
        self.client  = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistence ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get o create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG "},
                configuration={"hnsw":{"space":"cosine"}}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[any], embeddings: np.ndarray):
        """
        Add documents and embeddings to ChromaDB

        Args:
            documents:List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        #prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        #Determine the maximum batch size supported by ChromaDB
        max_batch_size = self.client.get_max_batch_size()
        batch_size = min(5000,max_batch_size)

        print(f"Batch size: {batch_size}")

        # Add to collection
        try:
            for start in range(0, len(documents),batch_size):
                end = min(start + batch_size, len(documents))

                self.collection.add(
                    ids=ids[start:end],
                    embeddings=embeddings_list[start:end],
                    metadatas = metadatas[start:end],
                    documents=documents_text[start:end]
                )
                print(f"Stored {end}/{len(documents)} documents")

            print(f"All documents stored Successfully")
            print(f"Total documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise 
vectorstore=VectorStore()
vectorstore

NameError: name 'List' is not defined

In [9]:
chunks

[Document(metadata={'producer': '3.0.36 (5.1.21)', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-21T18:15:49+00:00', 'title': 'LangChain Cheatsheet Carousel', 'moddate': '2026-06-21T20:20:17+02:00', 'source': '../data/pdf/langchain.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1', 'source_file': 'langchain.pdf', 'file_type': 'pdf'}, page_content='\uf5dc LANGCHAIN MASTERY\nThe Complete\nLangChain\nCheatsheet\nEvery component, every concept,\none real app built step-by-step.\nAI Coach John\n Follow \uf079 Repost'),
 Document(metadata={'producer': '3.0.36 (5.1.21)', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-21T18:15:49+00:00', 'title': 'LangChain Cheatsheet Carousel', 'moddate': '2026-06-21T20:20:17+02:00', 'source': '../data/pdf/langchain.pdf', 'total_pages': 28, 'page': 1

In [ ]:
# Extract text content from all document chunks into a list
texts=[doc.page_content for doc in chunks]

# Generate embeddings for each text chunk
embeddings=embedding_manager.generate_embeddings(texts)

# Store document chunks and embeddings in ChromaDB
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 25903 texts...


Batches:   0%|          | 0/810 [00:00<?, ?it/s]

Batches: 100%|██████████| 810/810 [19:53<00:00,  1.47s/it]


Generated embeddings with shape: (25903, 384)
Adding 25903 documents to vector store...
Batch size: 5000


In [1]:
#initialize the vectorDB when restart the notebook
vectorstore = VectorStore(
    collection_name="pdf_documents",
    persist_directory="../data/vector_store"
)

NameError: name 'VectorStore' is not defined

### Retriever Pipeline from Vector Store

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self, query:str, top_k:int=5, score_threshold:float=0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold:Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        
        if not query.strip():
            raise ValueError("Query cannot be empty")
        print(f"Retrieving documents for query: '{query}'")

        if top_k < 1:
            raise ValueError("top k must be greater than zero")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        collection_count = self.vector_store.collection.count()

        if collection_count == 0:
            return []

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=min(top_k,collection_count)
            )

            #Process results
            retrieved_docs = []

            if results['documents'] and results ['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score  >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content':document,
                            'metadata':metadata or {},
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [12]:
rag_retriever.retrieve("WHAT ARE AI AGENTS?")

Retrieving documents for query: 'WHAT ARE AI AGENTS?'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.72s/it]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_cea8cc28_3133',
  'content': 'WHAT ARE AI \nAGENTS?\n01 \nCHAPTER',
  'metadata': {'page_label': '8',
   'doc_index': 3133,
   'source': '../data/pdf/masteringaiagents.pdf',
   'content_length': 32,
   'page': 7,
   'total_pages': 111,
   'producer': 'PyPDF',
   'creator': 'PyPDF',
   'creationdate': '',
   'source_file': 'masteringaiagents.pdf',
   'file_type': 'pdf'},
  'similarity_score': 0.8913055658340454,
  'distance': 0.10869443416595459,
  'rank': 1},
 {'id': 'doc_2bafb736_3913',
  'content': 'WHAT ARE AI \nAGENTS?\n01 \nCHAPTER',
  'metadata': {'page': 7,
   'page_label': '8',
   'file_type': 'pdf',
   'creationdate': '',
   'content_length': 32,
   'source': '../data/pdf/masteringaiagents.pdf',
   'total_pages': 111,
   'producer': 'PyPDF',
   'doc_index': 3913,
   'creator': 'PyPDF',
   'source_file': 'masteringaiagents.pdf'},
  'similarity_score': 0.8913055658340454,
  'distance': 0.10869443416595459,
  'rank': 2},
 {'id': 'doc_7bef15e1_4303',
  'content': 'WHA

### Integration Vectordb context pipeline with llm output

In [13]:
# simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### initialize the GROK LLM 
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

#2. Simple RAG function: retrieve context + generate response 
def rag_simple(query,retriever,llm,top_k=3):
    #retriever the context
    results =  retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question." 

    # generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [14]:
answer=rag_simple("WHAT ARE AI AGENTS?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'WHAT ARE AI AGENTS?'
Top k: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


AI agents are software entities that perceive their environment, process information, and take actions autonomously to achieve specific goals.


### Enhanced RAG Pipeline Features

In [16]:
#Enhanced RAG pipeline  features
def rag_advanced(query, retriever, llm, top_k=5,min_score=0.2,return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answers, sources, confidence score, and optionally full context. 
    """
    results= retriever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
        return {'answer':'No relevant context found.','sources':[],'confidence':0.0,'context':''}
    
    #prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:300] + '...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer: """
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources':sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

#Example usage:
result = rag_advanced("WHAT ARE AI AGENTS?",rag_retriever,llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:",result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context preview:",result['context'][:300])

Retrieving documents for query: 'WHAT ARE AI AGENTS?'
Top k: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: AI agents are software systems that perceive their environment through data inputs, make decisions using artificial‑intelligence techniques (such as machine learning, reasoning, or planning), and act autonomously to achieve specific goals.
Sources: [{'source': 'masteringaiagents.pdf', 'page': 7, 'score': 0.8913055658340454, 'preview': 'WHAT ARE AI \nAGENTS?\n01 \nCHAPTER...'}, {'source': 'masteringaiagents.pdf', 'page': 7, 'score': 0.8913055658340454, 'preview': 'WHAT ARE AI \nAGENTS?\n01 \nCHAPTER...'}, {'source': 'masteringaiagents.pdf', 'page': 7, 'score': 0.8913055658340454, 'preview': 'WHAT ARE AI \nAGENTS?\n01 \nCHAPTER...'}]
Confidence: 0.8913055658340454
Context preview: WHAT ARE AI 
AGENTS?
01 
CHAPTER

WHAT ARE AI 
AGENTS?
01 
CHAPTER

WHAT ARE AI 
AGENTS?
01 
CHAPTER


In [17]:
#---Advanced RAG pipeline: Streaming, Citations, history, Summarization ----
from typing import List,Dict,Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever,llm):
        self.retriever = retriever
        self.llm=llm
        self.history = [] #Store query histry
    
    def query(self, question:str, top_k:int = 5, min_score:float = 0.2, stream:bool = False, summarize:bool = False ) ->  Dict[str,Any]:
        #Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k,score_threshold=min_score)    
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
                'page':doc['metadata'].get('page','unknown'),
                'score':doc['similarity_score'],
                'preview':doc['content'][:300] + '...'
            } for doc in results]
            #streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion:{question}\n\nAnswer:"""

            if stream:
                print("Streaming answer:\n")            
                chunks = []
                for chunk in self.llm.stream(prompt):
                    text = chunk.content
                    print(text,end='',flush=True)
                    chunks.append(text)
                    
                print()
                answer = "".join(chunks)
            else:
                response = self.llm.invoke(prompt)
                answer = response.content

        #Add citations to answer
        citations = [f"[{i+1}]{src['source']} (page{src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        #Optionally sumarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        #Store query history
        self.history.append({
            'question': question,
            'answer':answer,
            'sources':sources,
            'summary':summary
        })

        return {
            'question':question,
            'answer':answer_with_citations,
            'sources':sources,
            'summary':summary,
            'history':self.history
        }

#Example usage
adv_rag = AdvancedRAGPipeline(rag_retriever,llm)
result = adv_rag.query("WHAT ARE AI AGENTS?",top_k=3, min_score=0.3,stream=True,summarize=True)
print("\nFinal Answer:",result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'WHAT ARE AI AGENTS?'
Top k: 3, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:



AI agents are software systems that perceive their environment through data inputs, make decisions using artificial‑intelligence techniques (such as machine learning, reasoning, or planning), and act autonomously to achieve specific goals.

Final Answer: AI agents are software systems that gather information from their environment via data inputs and use AI methods—like machine learning, reasoning, or planning—to make decisions. They then act autonomously to pursue defined objectives.
History: {'question': 'WHAT ARE AI AGENTS?', 'answer': 'AI agents are software systems that perceive their environment through data inputs, make decisions using artificial‑intelligence techniques (such as machine learning, reasoning, or planning), and act autonomously to achieve specific goals.', 'sources': [{'source': 'masteringaiagents.pdf', 'page': 7, 'score': 0.8913055658340454, 'preview': 'WHAT ARE AI \nAGENTS?\n01 \nCHAPTER...'}, {'source': 'masteringaiagents.pdf', 'page': 7, 'score': 0.891305565834